# Проект: нейросеть для предсказания калорийности блюд

## Этап 1. Исследовательский анализ данных (EDA)

**Цель:** понять, с чем мы работаем, и сформировать стратегию обучения.

**Что исследуем:**
1. Общая статистика по датасету (размер, train/test split)
2. Распределение целевой переменной (`total_calories`)
3. Распределение массы блюда и её связь с калорийностью
4. Ингредиенты: сколько их всего, как часто встречаются, сколько в одном блюде
5. Фотографии: размеры, каналы, примеры
6. Пропуски / некорректные значения

**На выходе:** решения по препроцессингу, аугментациям, архитектуре.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# === УКАЖИ ПУТЬ К ДАТАСЕТУ ===
DATA_DIR = Path('data')  # поменяй на свой путь, если датасет лежит в другом месте

assert DATA_DIR.exists(), f'Директория {DATA_DIR} не найдена'
print('Содержимое DATA_DIR:', os.listdir(DATA_DIR))

## 1. Загрузка таблиц

In [ ]:
dish = pd.read_csv(DATA_DIR / 'dish.csv')
ingredients = pd.read_csv(DATA_DIR / 'ingredients.csv')

print('dish.csv:', dish.shape)
print('ingredients.csv:', ingredients.shape)
dish.head()

In [ ]:
ingredients.head()

In [ ]:
# Типы данных и пропуски
print('=== dish.csv ===')
print(dish.dtypes)
print('\nПропуски:')
print(dish.isna().sum())
print('\n=== ingredients.csv ===')
print(ingredients.dtypes)
print('\nПропуски:')
print(ingredients.isna().sum())

In [ ]:
# Размер train/test
print('Распределение split:')
print(dish['split'].value_counts())
print(f"\nВсего блюд: {len(dish)}")
print(f"Уникальных ингредиентов: {len(ingredients)}")

## 2. Целевая переменная — `total_calories`

In [ ]:
print(dish['total_calories'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(dish['total_calories'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Распределение калорийности (все блюда)')
axes[0].set_xlabel('total_calories, ккал')

sns.boxplot(data=dish, x='split', y='total_calories', ax=axes[1])
axes[1].set_title('Калорийность по сплитам')
plt.tight_layout(); plt.show()

In [ ]:
# Сравним распределения train vs test — важно чтобы не было сильного сдвига
fig, ax = plt.subplots(figsize=(8, 4))
for split_name in dish['split'].unique():
    subset = dish[dish['split'] == split_name]['total_calories']
    sns.kdeplot(subset, label=f'{split_name} (n={len(subset)})', ax=ax)
ax.set_title('Плотность калорийности: train vs test')
ax.set_xlabel('total_calories')
ax.legend(); plt.show()

print('\nСтатистики по сплитам:')
print(dish.groupby('split')['total_calories'].describe())

## 3. Масса блюда и её связь с калорийностью

Масса — прямой предиктор калорийности. Проверим корреляцию.

In [ ]:
print(dish['total_mass'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(dish['total_mass'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Распределение массы порции')
axes[0].set_xlabel('total_mass, г')

sns.scatterplot(data=dish, x='total_mass', y='total_calories', 
                hue='split', alpha=0.4, s=15, ax=axes[1])
axes[1].set_title(f"Масса vs калории (corr={dish[['total_mass','total_calories']].corr().iloc[0,1]:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
# Baseline: простейшая модель — средняя плотность ккал/г
train_df = dish[dish['split'] == 'train']
test_df  = dish[dish['split'] == 'test']

# Средняя калорийность на грамм (по трейну)
avg_kcal_per_g = (train_df['total_calories'] / train_df['total_mass']).mean()
print(f'Средняя плотность ккал/г по train: {avg_kcal_per_g:.3f}')

# Предсказание baseline = mass * avg_kcal_per_g
baseline_pred = test_df['total_mass'] * avg_kcal_per_g
baseline_mae = (baseline_pred - test_df['total_calories']).abs().mean()
print(f'Baseline MAE (только масса × средняя плотность): {baseline_mae:.2f}')

# Ещё проще: предсказываем среднюю калорийность для всех
mean_cal = train_df['total_calories'].mean()
naive_mae = (test_df['total_calories'] - mean_cal).abs().mean()
print(f'Naive MAE (предсказание = средняя калорийность): {naive_mae:.2f}')

**Вывод:** эти два бейзлайна — наша точка отсчёта. Нейросеть должна обойти хотя бы baseline_mae (масса × средняя плотность).

## 4. Ингредиенты

In [ ]:
# Парсим список ID ингредиентов в каждом блюде
def parse_ingredients(s):
    if pd.isna(s) or s == '':
        return []
    # формат: ingr_0000000122;ingr_0000000026;...
    # ненулевая часть — это ID
    tokens = [t.strip() for t in str(s).split(';') if t.strip()]
    ids = []
    for t in tokens:
        # берём числовую часть после 'ingr_'
        if '_' in t:
            num = t.split('_')[-1].lstrip('0') or '0'
            ids.append(int(num))
        else:
            try:
                ids.append(int(t))
            except ValueError:
                pass
    return ids

dish['ingr_ids'] = dish['ingredients'].apply(parse_ingredients)
dish['n_ingredients'] = dish['ingr_ids'].apply(len)

print('Статистика по количеству ингредиентов в блюде:')
print(dish['n_ingredients'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(dish['n_ingredients'], bins=30, ax=axes[0])
axes[0].set_title('Кол-во ингредиентов в блюде')

sns.scatterplot(data=dish, x='n_ingredients', y='total_calories', alpha=0.3, s=15, ax=axes[1])
axes[1].set_title('Число ингредиентов vs калорийность')
plt.tight_layout(); plt.show()

In [ ]:
# Топ-20 самых частых ингредиентов
from collections import Counter

all_ids = [i for ids in dish['ingr_ids'] for i in ids]
counter = Counter(all_ids)
print(f'Уникальных ингредиентов, встречающихся в блюдах: {len(counter)}')
print(f'Всего в справочнике ingredients.csv: {len(ingredients)}')

# Мапим id -> название
id_to_name = dict(zip(ingredients['id'].astype(int), ingredients['ingr']))

top20 = counter.most_common(20)
top20_df = pd.DataFrame(top20, columns=['id', 'count'])
top20_df['name'] = top20_df['id'].map(id_to_name)
top20_df

## 5. Фотографии

In [ ]:
IMAGES_DIR = DATA_DIR / 'images'

# Проверим что для каждого dish_id есть папка с rgb.png
sample_ids = dish['dish_id'].sample(min(200, len(dish)), random_state=42).tolist()
missing = []
sizes = []
for did in tqdm(sample_ids, desc='Проверка фото'):
    p = IMAGES_DIR / str(did) / 'rgb.png'
    if not p.exists():
        missing.append(did)
    else:
        with Image.open(p) as img:
            sizes.append(img.size + (img.mode,))

print(f'Отсутствующих файлов в выборке 200: {len(missing)}')
if sizes:
    sizes_df = pd.DataFrame(sizes, columns=['width', 'height', 'mode'])
    print('\nРазмеры (на выборке 200):')
    print(sizes_df[['width','height']].describe())
    print('\nМоды каналов:', sizes_df['mode'].value_counts().to_dict())

In [ ]:
# Посмотрим несколько случайных примеров
sample = dish.sample(12, random_state=42).reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i, row in sample.iterrows():
    ax = axes[i // 4, i % 4]
    p = IMAGES_DIR / str(row['dish_id']) / 'rgb.png'
    try:
        img = Image.open(p)
        ax.imshow(img)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
    ax.set_title(f"{row['dish_id']}\n{row['total_calories']:.0f} ккал / {row['total_mass']:.0f} г", fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Посмотрим на выбросы

In [ ]:
# Самые калорийные и самые лёгкие
print('Топ-5 самых калорийных:')
print(dish.nlargest(5, 'total_calories')[['dish_id','total_calories','total_mass','n_ingredients']])
print('\nТоп-5 с самой низкой калорийностью:')
print(dish.nsmallest(5, 'total_calories')[['dish_id','total_calories','total_mass','n_ingredients']])

# Странные случаи: нулевая масса или нулевые калории
print('\nБлюд с total_mass=0:', (dish['total_mass'] == 0).sum())
print('Блюд с total_calories=0:', (dish['total_calories'] == 0).sum())
print('Блюд без ингредиентов:', (dish['n_ingredients'] == 0).sum())

## 7. Выводы EDA и план решения

_Заполни на основе того, что увидел выше:_

**Что важно:**
- **Масса сильно коррелирует с калориями** → обязательно используем как отдельный признак.
- **Ингредиенты** → используем как multi-hot или embedding+mean.
- **Фото** → для контекста (плотность калорий зависит от типа еды).
- **Распределение калорий** — если скошенное, можно попробовать `log(target)` или `target/mass` как альтернативную цель.

**Решения:**

| Решение | Обоснование |
|---|---|
| Мультимодальная модель (image + ingredients + mass) | Табличные фичи дают больше сигнала, чем фото в одиночку |
| Backbone: EfficientNet-B0 | Быстрый, на 4070 Ti влезает большой батч, хорошо работает на ImageNet-подобных данных |
| Resize 224×224 | Стандартный размер для ImageNet-бэкбонов |
| Нормализация ImageNet-статистиками | Совместимость с предобученными весами |
| Аугментации train: RandomResizedCrop, HFlip, ColorJitter | Еда на фото часто снята под разным освещением и ракурсом |
| Loss: SmoothL1 | Устойчив к выбросам, на последних эпохах можно перейти на L1 |
| Нормализация массы: z-score по train | Чтобы MLP не страдал от разного масштаба фичей |
| Ингредиенты: nn.Embedding + mean | Компактное представление переменной длины |
| Метрики: MAE (целевая), RMSE, R² (для понимания) | MAE — из ТЗ |

---

## Этап 2. Пайплайн обучения — smoke test

Код пайплайна лежит в `scripts/`. Здесь только импортируем и проверяем, что:
1. DataLoader-ы строятся без ошибок и достают данные.
2. Модель строится и делает forward на одном батче.
3. Тренировочный цикл проходит 1 эпоху без падения.

После успешного smoke-теста — переходим к полному обучению (Этап 3).

In [ ]:
# Автоперезагрузка модулей при правках в .py-файлах
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# Делаем scripts/ импортируемым
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import load_config
from scripts.utils import set_seed, train
from scripts.dataset import build_dataloaders
from scripts.model import build_model
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1) Загружаем конфиг и проверяем DataLoader-ы
cfg = load_config('configs/config.yaml')
set_seed(cfg.seed, deterministic=cfg.deterministic)

data = build_dataloaders(cfg, seed=cfg.seed)
print(f"train={data['train_size']}  val={data['val_size']}  test={data['test_size']}")
print(f"mass mean={data['mass_mean']:.2f}  std={data['mass_std']:.2f}")

# Достанем один батч и посмотрим на содержимое
batch = next(iter(data['train_loader']))
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v).__name__} (len={len(v)})")

In [ ]:
# 2) Проверяем forward модели на этом батче
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg).to(device)
model.eval()

with torch.inference_mode():
    raw = model(
        batch['image'].to(device),
        batch['ingr_ids'].to(device),
        batch['ingr_mask'].to(device),
        batch['mass_norm'].to(device),
    )
print('raw output shape:', raw.shape)
print('sample raw values (density):', raw[:5].cpu().tolist())
print('sample mass:', batch['mass'][:5].tolist())
print('sample target calories:', batch['calories'][:5].tolist())

# Освобождаем память для следующего шага
del model, raw
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 3) Smoke-test: запускаем train() на 2 эпохи. Замеряем время.
# Если всё ок — переходим к Этапу 3 с полными настройками.
import copy, time
smoke_cfg = copy.deepcopy(cfg)
smoke_cfg.train.epochs = 2
smoke_cfg.train.early_stopping_patience = 999  # не хотим раннюю остановку на тесте
smoke_cfg.paths.best_model_name = 'smoke_best.pth'
smoke_cfg.paths.last_model_name = 'smoke_last.pth'

t0 = time.time()
result = train(smoke_cfg)
print(f"\nSMOKE TEST DONE in {(time.time()-t0)/60:.1f} min")
print(f"Best val MAE: {result['best_val_mae']:.2f}")
print(f"\nЕсли цифра уже близка к baseline (~104) — пайплайн работает корректно.")
print(f"Время одной эпохи ≈ {(time.time()-t0)/2/60:.2f} мин — ориентир для Этапа 3.")

---

## Этап 3. Полное обучение модели

Запускаем `train()` с полным конфигом (30 эпох, early stopping patience=8).

**Ожидания по времени:**
- GPU (4070 Ti): ~5–10 минут на полный train
- CPU: ~60 минут (может закончиться раньше из-за early stopping)

**Результат:**
- Чекпоинт `checkpoints/best_model.pth` с лучшими весами по val MAE
- История обучения в `result['history']`
- Графики train/val loss и MAE

In [ ]:
# Полный train
import time

full_cfg = load_config('configs/config.yaml')
print(f"Запуск обучения: {full_cfg.train.epochs} эпох, backbone={full_cfg.model.backbone}")
print(f"LR: head={full_cfg.train.lr_head}, backbone={full_cfg.train.lr_backbone}")
print(f"Loss: {full_cfg.train.loss}, predict_density={full_cfg.train.predict_density}")
print()

t0 = time.time()
result = train(full_cfg)
elapsed_min = (time.time() - t0) / 60

print(f"\n{'='*60}")
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО за {elapsed_min:.1f} мин")
print(f"Best val MAE = {result['best_val_mae']:.2f}")
print(f"Best ckpt:  {result['best_ckpt_path']}")
print(f"Last ckpt:  {result['last_ckpt_path']}")
print(f"{'='*60}")

### Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

history = result['history']
epochs_range = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='train', marker='o', markersize=4)
axes[0].plot(epochs_range, history['val_loss'], label='val', marker='s', markersize=4)
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss (SmoothL1, в пространстве density)')
axes[0].set_title('Loss по эпохам')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE
axes[1].plot(epochs_range, history['train_mae'], label='train', marker='o', markersize=4)
axes[1].plot(epochs_range, history['val_mae'], label='val', marker='s', markersize=4)
axes[1].axhline(50, color='green', linestyle='--', alpha=0.6, label='target (<50)')
axes[1].axhline(104.12, color='red', linestyle='--', alpha=0.6, label='baseline (104.12)')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('MAE (ккал)')
axes[1].set_title('MAE по эпохам (исходная шкала)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Числовая сводка
print(f"Эпох выполнено:          {len(epochs_range)}")
print(f"Лучший val MAE:          {min(history['val_mae']):.2f} (эпоха {history['val_mae'].index(min(history['val_mae'])) + 1})")
print(f"Финальный val MAE:       {history['val_mae'][-1]:.2f}")
print(f"Финальный train MAE:     {history['train_mae'][-1]:.2f}")
print(f"Gap train-val (overfit): {history['val_mae'][-1] - history['train_mae'][-1]:.2f}")

### Информация о сохранённом чекпоинте

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path(result['best_ckpt_path'])
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

print(f"Путь:                  {ckpt_path}")
print(f"Размер файла:          {ckpt_path.stat().st_size / 1e6:.1f} MB")
print(f"Эпоха сохранения:      {ckpt['epoch']}")
print(f"Val MAE:               {ckpt['val_mae']:.2f}")
print(f"mass_mean (по train):  {ckpt['mass_mean']:.2f}")
print(f"mass_std (по train):   {ckpt['mass_std']:.2f}")
print(f"Ключей в model_state:  {len(ckpt['model_state_dict'])}")

# Оценка статуса
if ckpt['val_mae'] < 50:
    print("\n✅ Цель MAE < 50 на val ДОСТИГНУТА. Переходим к Этапу 4 — валидации на test.")
elif ckpt['val_mae'] < 70:
    print("\n⚠️  Близко, но не дотянули. Варианты доработки:")
    print("   - Увеличить epochs до 50")
    print("   - Попробовать convnext_tiny / efficientnet_b3 как бэкбон")
    print("   - Снизить lr_backbone до 5e-5")
else:
    print("\n⚠️  Цель пока не достигнута — нужна более серьёзная доработка.")

---

## Этап 4. Валидация качества на test

**По ТЗ:**
1. Загрузить обученную модель и запустить инференс на test.
2. Вывести целевую метрику MAE.
3. Найти топ-5 блюд с наибольшей ошибкой, визуализировать их и описать возможные причины.

### 4.1. Загрузка модели и инференс на test

In [ ]:
import torch
import numpy as np
from scripts.config import load_config
from scripts.utils import load_model_from_checkpoint, predict_loader, set_seed
from scripts.dataset import build_dataloaders

cfg = load_config('configs/config.yaml')
set_seed(cfg.seed, deterministic=cfg.deterministic)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# Загружаем модель из best-чекпоинта
ckpt_path = f"{cfg.paths.checkpoints_dir}/{cfg.paths.best_model_name}"
model, ckpt = load_model_from_checkpoint(ckpt_path, cfg, device)
print(f'Loaded from: {ckpt_path}')
print(f'Epoch: {ckpt["epoch"]},  val MAE at save: {ckpt["val_mae"]:.2f}')
print(f'mass_mean: {ckpt["mass_mean"]:.2f},  mass_std: {ckpt["mass_std"]:.2f}')

# Готовим даталоадеры — нужен test_loader
data = build_dataloaders(cfg, seed=cfg.seed)
test_loader = data['test_loader']
print(f'test size: {data["test_size"]}')

In [ ]:
# Инференс на test
import time
t0 = time.time()
dish_ids, preds, targets, masses = predict_loader(
    model, test_loader, device, predict_density=cfg.train.predict_density
)
print(f'Инференс: {time.time()-t0:.1f} s, {len(dish_ids)} блюд')

# Финальные метрики
errors = preds - targets
abs_errors = np.abs(errors)

test_mae = abs_errors.mean()
test_rmse = np.sqrt((errors ** 2).mean())
test_mape = (abs_errors / np.clip(targets, 1.0, None)).mean() * 100

# Baseline для сравнения
baseline_pred = masses * 1.280  # avg kcal/g по train (из EDA)
baseline_mae = np.abs(baseline_pred - targets).mean()

print('\n' + '=' * 60)
print('ФИНАЛЬНЫЕ МЕТРИКИ НА TEST')
print('=' * 60)
print(f'Test MAE:      {test_mae:.2f} ккал  (цель: < 50 {"✅" if test_mae < 50 else "❌"})')
print(f'Test RMSE:     {test_rmse:.2f} ккал')
print(f'Test MAPE:     {test_mape:.1f} %')
print(f'Baseline MAE:  {baseline_mae:.2f} ккал  (mass × avg density)')
print(f'Улучшение:     {(baseline_mae - test_mae) / baseline_mae * 100:.1f}% лучше baseline')
print('=' * 60)

### 4.2. Распределение ошибок

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1) Распределение абсолютных ошибок
sns.histplot(abs_errors, bins=40, kde=True, ax=axes[0])
axes[0].axvline(test_mae, color='red', linestyle='--', label=f'MAE={test_mae:.1f}')
axes[0].axvline(50, color='green', linestyle='--', label='target=50')
axes[0].set_xlabel('|error|, ккал')
axes[0].set_title('Распределение абсолютных ошибок')
axes[0].legend()

# 2) Предсказание vs истинное
axes[1].scatter(targets, preds, alpha=0.4, s=15)
lims = [0, max(targets.max(), preds.max()) * 1.05]
axes[1].plot(lims, lims, 'r--', label='y=x (идеал)')
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel('истинное, ккал')
axes[1].set_ylabel('предсказание, ккал')
axes[1].set_title('Predicted vs Actual')
axes[1].legend()

# 3) Ошибка по массе — есть ли систематика
axes[2].scatter(masses, errors, alpha=0.4, s=15)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('масса блюда, г')
axes[2].set_ylabel('error = pred - true, ккал')
axes[2].set_title('Ошибка vs масса блюда')

plt.tight_layout()
plt.show()

print(f'Медиана |error|:   {np.median(abs_errors):.2f} ккал')
print(f'90-й перцентиль:   {np.percentile(abs_errors, 90):.2f} ккал')
print(f'95-й перцентиль:   {np.percentile(abs_errors, 95):.2f} ккал')
print(f'Max |error|:       {abs_errors.max():.2f} ккал')

### 4.3. Топ-5 самых тяжёлых примеров

По ТЗ: визуализация и анализ возможных причин ошибок.

In [ ]:
import pandas as pd
from pathlib import Path

# Собираем все метаданные в DataFrame
results_df = pd.DataFrame({
    'dish_id': dish_ids,
    'mass': masses,
    'true_calories': targets,
    'pred_calories': preds,
    'error': errors,
    'abs_error': abs_errors,
    'density_true': targets / np.clip(masses, 1.0, None),
    'density_pred': preds / np.clip(masses, 1.0, None),
})

# Подтягиваем ингредиенты
dish_df = pd.read_csv(cfg.paths.dish_csv)
ingredients_df = pd.read_csv(cfg.paths.ingredients_csv)
id_to_name = dict(zip(ingredients_df['id'].astype(int), ingredients_df['ingr']))

def get_ingr_names(dish_id_str):
    row = dish_df[dish_df['dish_id'] == dish_id_str]
    if len(row) == 0:
        return []
    from scripts.dataset import parse_ingredients
    ids = parse_ingredients(row.iloc[0]['ingredients'])
    return [id_to_name.get(i, f'?id={i}') for i in ids]

top5 = results_df.nlargest(5, 'abs_error').reset_index(drop=True)
print('ТОП-5 блюд с наибольшей ошибкой модели:')
print()
for i, row in top5.iterrows():
    ingr_names = get_ingr_names(row['dish_id'])
    print(f"{i+1}. {row['dish_id']}")
    print(f"   истина:     {row['true_calories']:7.1f} ккал,  масса {row['mass']:6.1f} г,  плотность {row['density_true']:.2f} ккал/г")
    print(f"   предикт:    {row['pred_calories']:7.1f} ккал,                    плотность {row['density_pred']:.2f} ккал/г")
    print(f"   ошибка:     {row['error']:+.1f} ккал  (|err|={row['abs_error']:.1f})")
    print(f"   ингредиенты ({len(ingr_names)}): {', '.join(ingr_names[:10])}{' ...' if len(ingr_names) > 10 else ''}")
    print()

In [ ]:
# Визуализация топ-5 с фото
from PIL import Image

images_dir = Path(cfg.paths.images_dir)
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for i, row in top5.iterrows():
    ax = axes[i]
    img_path = images_dir / row['dish_id'] / 'rgb.png'
    try:
        img = Image.open(img_path)
        ax.imshow(img)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
    title = (
        f"#{i+1}  mass={row['mass']:.0f}г\n"
        f"true: {row['true_calories']:.0f} | pred: {row['pred_calories']:.0f}\n"
        f"error: {row['error']:+.0f} ккал"
    )
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4.4. Анализ причин ошибок

_Заполни на основе того, что увидел в топ-5. Ниже — шаблон причин, которые обычно встречаются:_

**Вероятные причины низкого качества на этих примерах:**

1. **Аномальная плотность калорий** (`density_true` сильно отличается от медианы ≈1.16 ккал/г):
   - Очень калорийные блюда — жареное с маслом, сыры, десерты с сахаром. Модель занижает, т.к. видит «обычную» еду.
   - Очень «пустые» блюда — салаты, супы, напитки. Модель завышает, ожидая средней плотности.

2. **Редкие или уникальные ингредиенты** — модель обучена на 200 реальных ингредиентах, но у каждого блюда своё сочетание. Если блюдо содержит редкие комбинации (встречались <5 раз в train), embedding не сформировал для них хорошее представление.

3. **Визуальная неоднозначность фото** — ракурс сверху вниз, плохое освещение, блик, полупрозрачный соус, крышка на контейнере. Image-encoder получает слабый сигнал.

4. **Экстремальные массы** — самые тяжёлые блюда (>800 г) в train малочисленны, модель экстраполирует плохо.

5. **Несоответствие фото и ингредиентов** — в датасете могут быть ошибки аннотации: фото одного блюда, а ингредиенты другого. Это фундаментальный шум, который не исправить моделированием.

6. **Блюда из нескольких частей/порций** — на фото тарелка, но в total_mass учтены соусы/гарниры, которые визуально занимают мало места.

**Возможные улучшения модели для этих сложных случаев:**
- Сильнее аугментировать редкие высококалорийные блюда (weighted sampler).
- Более мощный бэкбон (ConvNeXt-Tiny / EfficientNet-B3) для извлечения тонких визуальных признаков.
- Добавить логарифмирование таргета — сейчас MSE/SmoothL1 штрафует ошибки в ккал одинаково независимо от масштаба блюда, что смещает обучение в сторону «средних» блюд.

### 4.5. Итоги проекта

In [ ]:
print('=' * 60)
print('ИТОГИ ПРОЕКТА')
print('=' * 60)
print()
print('Архитектура:         мультимодальная (image + ingredients + mass)')
print(f'Бэкбон:              {cfg.model.backbone} (pretrained ImageNet)')
print(f'Параметров:          {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M')
print(f'Loss:                {cfg.train.loss}  (в пространстве density)')
print()
print('Данные:')
print(f'  train / val / test: {data["train_size"]} / {data["val_size"]} / {data["test_size"]}')
print()
print('Метрики:')
print(f'  Naive MAE:         168.52 ккал  (предсказание = средняя калорийность)')
print(f'  Baseline MAE:      104.12 ккал  (mass × avg density)')
print(f'  Model val MAE:     {ckpt["val_mae"]:.2f} ккал  (best checkpoint)')
print(f'  Model test MAE:    {test_mae:.2f} ккал  ← целевая метрика')
print(f'  Model test RMSE:   {test_rmse:.2f} ккал')
print()
status = '✅ ЦЕЛЬ MAE < 50 ДОСТИГНУТА' if test_mae < 50 else '❌ цель не достигнута'
print(status)